In [1]:
# ── Install / upgrade all dependencies ────────────────────────────────────────
%uv pip install -q \
    "chromadb>=0.5.3" \
    sentence-transformers \
    rank_bm25 \
    torch \
    transformers \
    peft \
    accelerate \
    sentencepiece \
    hf_transfer \
    huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

ROOT = Path("/mnt/s3/s3hm/s3hm")

MODEL_KEY        = "bge_m3"
MODEL_ID         = "BAAI/bge-m3"
QUERY_PREFIX     = None  # E5 convention: prepend 'query: ' at retrieval time
CHUNK_TYPE       = "structure_aware"
COLLECTION_NAME  = f"s3hm_{MODEL_KEY}_{CHUNK_TYPE}"

EMBED_DIR  = ROOT / "embedding"  / MODEL_KEY / CHUNK_TYPE

In [3]:
S3_STORE_DIR = ROOT / "chroma_vector_store" / MODEL_KEY / CHUNK_TYPE
STORE_DIR = Path("/tmp/chroma_vector_store") / MODEL_KEY / CHUNK_TYPE

In [4]:
from pathlib import Path
import shutil
import chromadb

# Remove an old incomplete local copy
if STORE_DIR.exists():
    shutil.rmtree(LOCAL_STORE_DIR)

STORE_DIR.parent.mkdir(parents=True, exist_ok=True)

# Copy the complete Chroma database from S3 to local disk
shutil.copytree(S3_STORE_DIR, STORE_DIR)

print("Local files:")
for item in STORE_DIR.rglob("*"):
    print(item)

client = chromadb.PersistentClient(
    path=str(STORE_DIR)
)

collection = client.get_collection(
    name="s3hm_bge_m3_structure_aware"
)

print("Collection count:", collection.count())

Local files:
/tmp/chroma_vector_store/bge_m3/structure_aware/850e5333-994c-4c19-8cb2-d78b82df864b
/tmp/chroma_vector_store/bge_m3/structure_aware/chroma.sqlite3
/tmp/chroma_vector_store/bge_m3/structure_aware/850e5333-994c-4c19-8cb2-d78b82df864b/data_level0.bin
/tmp/chroma_vector_store/bge_m3/structure_aware/850e5333-994c-4c19-8cb2-d78b82df864b/header.bin
/tmp/chroma_vector_store/bge_m3/structure_aware/850e5333-994c-4c19-8cb2-d78b82df864b/index_metadata.pickle
/tmp/chroma_vector_store/bge_m3/structure_aware/850e5333-994c-4c19-8cb2-d78b82df864b/length.bin
/tmp/chroma_vector_store/bge_m3/structure_aware/850e5333-994c-4c19-8cb2-d78b82df864b/link_lists.bin
Collection count: 1758


In [5]:
from pathlib import Path

print(f"Model      : {MODEL_ID}")
print(f"Chunk type : {CHUNK_TYPE}")
print(f"Collection : {COLLECTION_NAME}")
print(f"Store path : {STORE_DIR}")
print(f"Store exists: {(STORE_DIR / 'chroma.sqlite3').exists()}")

Model      : BAAI/bge-m3
Chunk type : structure_aware
Collection : s3hm_bge_m3_structure_aware
Store path : /tmp/chroma_vector_store/bge_m3/structure_aware
Store exists: True


In [12]:
# ── Core imports and abstract base classes ────────────────────────────────────
from __future__ import annotations
from abc import ABC, abstractmethod
from pathlib import Path
from typing import Any, Iterable, Sequence, Mapping, Optional, Callable
import copy
import hashlib
import unicodedata
from dataclasses import dataclass, field
from sentence_transformers import SentenceTransformer
import json


@dataclass(frozen=True)
class Chunk:
    id: str
    text: str
    metadata: Mapping[str, Any] = field(default_factory=dict)


@dataclass(frozen=True)
class Retrieved(Chunk):
    score: float = 0.0
    rank: Optional[int] = None


class DocumentLoader(ABC):
    @abstractmethod
    def __init__(self, pdf_dir: str | Path, config_path: str | Path): ...

    @abstractmethod
    def load(self) -> list[Any]: ...


class DocumentProcessor(ABC):
    @abstractmethod
    def process(self, text: str) -> str: ...

    def process_documents(self, documents, *, in_place=True, show_progress=True):
        docs = documents if in_place else copy.deepcopy(documents)

        iterator = docs
        if show_progress:
            try:
                from tqdm import tqdm

                iterator = tqdm(docs, desc="Processing documents")
            except Exception:
                pass

        for doc in iterator:
            if hasattr(doc, "page_content") and isinstance(doc.page_content, str):
                doc.page_content = self.process(doc.page_content)

        return docs


class Chunker(ABC):
    @abstractmethod
    def split(self, docs: Iterable[Any]) -> list[Any]: ...


class Embedder(ABC):
    @abstractmethod
    def embed_documents(self, texts: Sequence[str]) -> list[list[float]]: ...

    @abstractmethod
    def embed_query(self, text: str) -> list[float]: ...


class VectorStore(ABC):
    @abstractmethod
    def upsert(
        self, chunks: Sequence[Chunk], embeddings: Sequence[Sequence[float]]
    ) -> None: ...

    @abstractmethod
    def query(self, query_embedding: Sequence[float], top_k: int) -> list[dict]: ...


class Retriever(ABC):
    @abstractmethod
    def retrieve(
        self, query: str, top_k: int = 5, min_score: float = 0.0
    ) -> list[Retrieved]: ...


# ── ChromaVectorStore ─────────────────────────────────────────────────────────
class ChromaVectorStore(VectorStore):
    def __init__(self, *, persist_dir: str, collection_name: str = "s3hm"):
        import chromadb
        self.persist_dir     = str(Path(persist_dir))
        self.collection_name = collection_name
        self._client     = chromadb.PersistentClient(path=self.persist_dir)
        self._collection = self._client.get_or_create_collection(
            name=self.collection_name, metadata={"hnsw:space": "cosine"}
        )

    def upsert(self, chunks: Sequence[Chunk], embeddings: Sequence[Sequence[float]]) -> None:
        self._collection.upsert(
            ids=[c.id for c in chunks],
            documents=[c.text for c in chunks],
            metadatas=[dict(c.metadata) for c in chunks],
            embeddings=[list(map(float, e)) for e in embeddings],
        )

    def query(self, query_embedding: Sequence[float], top_k: int) -> list[dict]:
        res   = self._collection.query(
            query_embeddings=[list(map(float, query_embedding))],
            n_results=top_k,
            include=["documents", "metadatas", "distances"],
        )
        ids   = (res.get("ids")       or [[]])[0]
        docs  = (res.get("documents") or [[]])[0]
        metas = (res.get("metadatas") or [[]])[0]
        dists = (res.get("distances") or [[]])[0]
        return [
            {"id": ids[i], "text": docs[i], "metadata": metas[i] or {},
             "distance": float(dists[i])}
            for i in range(min(len(ids), len(docs), len(metas), len(dists)))
        ]

print("ChromaVectorStore defined")


# ── QueryEmbedder ─────────────────────────────────────────────────────────────
class QueryEmbedder(Embedder):
    """Query-only embedder, with encoded queries cached.

    A single question is embedded more than once per turn: retrieve_context(
    mode="hybrid") goes through DenseRetriever, and knowledge_boundary_detection
    then retrieves the same query again with mode="dense". Caching makes the
    repeats free.
    """

    def __init__(self, model_id: str, *, query_prefix: str | None = None,
                 encode_kwargs: dict | None = None, trust_remote_code: bool = False):
        self.query_prefix   = query_prefix
        self._encode_kwargs = encode_kwargs or {}
        self._model = SentenceTransformer(model_id, trust_remote_code=trust_remote_code)
        self._cache: dict[str, list[float]] = {}

    def embed_query(self, text: str) -> list[float]:
        cached = self._cache.get(text)
        if cached is not None:
            return cached
        t   = (self.query_prefix + text) if self.query_prefix else text
        vec = self._model.encode([t], normalize_embeddings=True,
                                 show_progress_bar=False, **self._encode_kwargs)
        embedding = vec[0].tolist()
        self._cache[text] = embedding
        return embedding

    def clear_cache(self) -> None:
        self._cache.clear()

    def embed_documents(self, texts):
        raise NotImplementedError("Use precomputed embeddings for document indexing")

print("QueryEmbedder defined")


# ── DenseRetriever ────────────────────────────────────────────────────────────
class DenseRetriever(Retriever):
    def __init__(self, *, embedder: Embedder, vector_store: VectorStore):
        self.embedder     = embedder
        self.vector_store = vector_store

    def retrieve(self, query: str, top_k: int = 5, min_score: float = 0.0) -> list[Retrieved]:
        if not isinstance(query, str) or not query.strip():
            return []
        qvec = self.embedder.embed_query(query.strip())
        rows = self.vector_store.query(query_embedding=qvec, top_k=top_k) or []
        results: list[Retrieved] = []
        for rank, row in enumerate(rows, 1):
            score = 1.0 - float(row.get("distance", 1.0))
            if score < min_score:
                continue
            results.append(Retrieved(
                id=str(row.get("id", "")), text=str(row.get("text", "")),
                metadata=dict(row.get("metadata") or {}), score=score, rank=rank,
            ))
        return results

print("DenseRetriever defined")


# ── KeyWordRetriever (BM25) ───────────────────────────────────────────────────
def _tokenize_si(text: str) -> list[str]:
    if text is None:
        return []
    text = unicodedata.normalize("NFC", str(text)).replace("\u200d", "").replace("\u200c", "")
    cleaned = []
    for ch in text:
        cat = unicodedata.category(ch)
        cleaned.append(" " if cat.startswith("P") or cat.startswith("S") else ch)
    return [t for t in "".join(cleaned).lower().split() if t]


class KeyWordRetriever(Retriever):
    def __init__(self, documents, *, tokenizer: Callable[[str], list[str]] = _tokenize_si):
        self.tokenizer   = tokenizer
        self.ids: list[str]        = []
        self.text_by_id: dict[str, str]  = {}
        self.meta_by_id: dict[str, dict] = {}
        self.bm25 = None

        tokenized: list[list[str]] = []
        for i, doc in enumerate(list(documents)):
            content = (
                doc.get("text") if isinstance(doc, dict)
                else getattr(doc, "text", getattr(doc, "page_content", ""))
            ) or ""
            doc_id = (
                doc.get("id") if isinstance(doc, dict) else getattr(doc, "id", None)
            ) or hashlib.sha1(f"{i}:{content}".encode()).hexdigest()
            self.ids.append(str(doc_id))
            self.text_by_id[str(doc_id)] = str(content)
            self.meta_by_id[str(doc_id)] = dict(
                doc.get("metadata", {}) if isinstance(doc, dict)
                else getattr(doc, "metadata", {}) or {}
            )
            tokenized.append(tokenizer(str(content)))

        if self.ids and any(tokenized):
            from rank_bm25 import BM25Okapi
            self.bm25 = BM25Okapi(tokenized)
            print(f"BM25 index built: {len(self.ids)} chunks")

    def search(self, query: str, *, top_k: int = 10) -> list[dict[str, Any]]:
        if not query.strip() or self.bm25 is None:
            return []
        q_tokens = self.tokenizer(query)
        if not q_tokens:
            return []
        scores  = self.bm25.get_scores(q_tokens)
        top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:min(top_k, len(self.ids))]
        return [
            {"id": self.ids[i], "content": self.text_by_id[self.ids[i]],
             "metadata": self.meta_by_id[self.ids[i]],
             "bm25_score": float(scores[i]), "rank": rank}
            for rank, i in enumerate(top_idx, 1)
        ]

    def retrieve(self, query: str, top_k: int = 5, min_score: float = 0.0) -> list[Retrieved]:
        return [
            Retrieved(id=r["id"], text=r["content"], metadata=dict(r["metadata"]),
                      score=r["bm25_score"], rank=r["rank"])
            for r in self.search(query, top_k=top_k)
            if r["bm25_score"] >= min_score
        ]

print("KeyWordRetriever defined")


# ── HybridRetriever (Reciprocal Rank Fusion) ──────────────────────────────────
class HybridRetriever(Retriever):
    def __init__(self, dense_retriever: Retriever, keyword_retriever,
                 *, w_dense: float = 1.0, w_bm25: float = 1.0, rrf_k: int = 60):
        self.dense_retriever   = dense_retriever
        self.keyword_retriever = keyword_retriever
        self.w_dense = float(w_dense)
        self.w_bm25  = float(w_bm25)
        self.rrf_k   = int(rrf_k)

    def retrieve(self, query: str, top_k: int = 5, min_score: float = 0.0,
                 *, dense_top_k: int = 20, bm25_top_k: int = 20) -> list[Retrieved]:
        if not isinstance(query, str) or not query.strip():
            return []
        dense = self.dense_retriever.retrieve(query, top_k=dense_top_k, min_score=0.0)
        if hasattr(self.keyword_retriever, "search"):
            sparse_rows = self.keyword_retriever.search(query, top_k=bm25_top_k) or []
        else:
            sparse_rows = [
                {"id": r.id, "content": r.text, "metadata": dict(r.metadata),
                 "bm25_score": r.score, "rank": r.rank}
                for r in self.keyword_retriever.retrieve(query, top_k=bm25_top_k, min_score=0.0)
            ]

        dense_by_id  = {r.id: r for r in dense if r.id}
        sparse_by_id = {str(r.get("id", "")): r for r in sparse_rows if r.get("id")}

        fused: dict[str, float] = {}
        for rank, r in enumerate(dense, 1):
            if r.id:
                fused[r.id] = fused.get(r.id, 0.0) + self.w_dense / (self.rrf_k + rank)
        for row in sparse_rows:
            doc_id = str(row.get("id", ""))
            if doc_id:
                fused[doc_id] = fused.get(doc_id, 0.0) + self.w_bm25 / (self.rrf_k + int(row.get("rank") or 1))

        results: list[Retrieved] = []
        for doc_id, hybrid_score in fused.items():
            if hybrid_score < min_score:
                continue
            src  = dense_by_id.get(doc_id) or sparse_by_id.get(doc_id)
            text = src.text if isinstance(src, Retrieved) else str((src or {}).get("content", ""))
            meta = dict(src.metadata if isinstance(src, Retrieved) else (src or {}).get("metadata", {}))
            results.append(Retrieved(id=doc_id, text=text, metadata=meta, score=float(hybrid_score)))

        results.sort(key=lambda x: x.score, reverse=True)
        return [
            Retrieved(id=r.id, text=r.text, metadata=r.metadata, score=r.score, rank=i)
            for i, r in enumerate(results[:top_k], 1)
        ]

print("HybridRetriever defined")


# ── Load chunk texts for BM25 ─────────────────────────────────────────────────
meta_file = EMBED_DIR / "chunks_meta.jsonl"
chunks: list[Chunk] = []
with meta_file.open("r", encoding="utf-8") as fh:
    for line in fh:
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        chunks.append(Chunk(
            id=str(row.get("id") or ""),
            text=str(row.get("text") or ""),
            metadata=dict(row.get("metadata") or {}),
        ))
print(f"Loaded {len(chunks)} chunks from {CHUNK_TYPE} index")

# ── Connect to Chroma vector store ────────────────────────────────────────────
vector_store = ChromaVectorStore(
    persist_dir=str(STORE_DIR),
    collection_name=COLLECTION_NAME,
)
print(f"Chroma collection '{COLLECTION_NAME}' connected")

# ── Load query embedder ───────────────────────────────────────────────────────
print(f"Loading {MODEL_ID} ...")
embedder = QueryEmbedder(
    model_id=MODEL_ID,
    query_prefix=QUERY_PREFIX,
)
print("Embedder ready")

# ── Build retrievers ──────────────────────────────────────────────────────────
dense_retriever   = DenseRetriever(embedder=embedder, vector_store=vector_store)
keyword_retriever = KeyWordRetriever(chunks)
hybrid_retriever  = HybridRetriever(dense_retriever, keyword_retriever, w_dense=1.0, w_bm25=1.0, rrf_k=60)

print("\nAll retrievers ready.")


# ── Retrive context ───────────────────────────────────────────────────────
def retrieve_context(
    query: str,
    *,
    top_k: int = 5,
    mode: str = "hybrid",      # "hybrid" | "dense" | "bm25"
    dense_pool: int = 20,
    bm25_pool: int = 20,
    min_score: float = 0.0,
) -> tuple[list[dict], str]:
    """Retrieve relevant context chunks for *query*.

    Returns
    -------
    results : list[dict]
        Each dict has keys: rank, score, text, grade, chapter, section, page, source.
    context_str : str
        Formatted string suitable for insertion into an LLM prompt.
    """
    if mode == "hybrid":
        raw: list[Retrieved] = hybrid_retriever.retrieve(
            query, top_k=top_k, min_score=min_score,
            dense_top_k=dense_pool, bm25_top_k=bm25_pool,
        )
    elif mode == "dense":
        raw = dense_retriever.retrieve(query, top_k=top_k, min_score=min_score)
    elif mode == "bm25":
        raw = keyword_retriever.retrieve(query, top_k=top_k, min_score=min_score)
    else:
        raise ValueError(f"mode must be 'hybrid', 'dense', or 'bm25', got {mode!r}")

    results: list[dict] = []
    for r in raw:
        meta = r.metadata or {}
        results.append({
            "rank":    r.rank,
            "score":   round(r.score, 6),
            "text":    r.text,
            "grade":   meta.get("grade", ""),
            "chapter": meta.get("toc_chapter_title", ""),
            "section": meta.get("toc_section_title", ""),
            "page":    meta.get("page", ""),
            "source":  meta.get("source_file", ""),
        })

    # Build LLM-ready context string
    blocks: list[str] = []
    for item in results:
        blocks.append(f"[{item['text']}]")

    context_str = " ".join(blocks)
    return results, context_str


print("retrieve_context() ready")

ChromaVectorStore defined
QueryEmbedder defined
DenseRetriever defined
KeyWordRetriever defined
HybridRetriever defined
Loaded 1758 chunks from structure_aware index
Chroma collection 's3hm_bge_m3_structure_aware' connected
Loading BAAI/bge-m3 ...
Embedder ready
BM25 index built: 1758 chunks

All retrievers ready.
retrieve_context() ready


In [6]:
import math
import re
from statistics import mean


def _boundary_tokens(text: str) -> set[str]:
    """Tokenize for the `overlap` feature.

    MUST stay identical to the tokenizer used when KB_MODEL was fit in
    threshold.ipynb, which uses `_tokenize_si`. The earlier regex here
    (`[\w඀-෿]+`) differed: it did not NFC-normalize and did not strip
    ZWJ/ZWNJ, so a conjunct such as ශ්‍ර was split into two tokens at inference
    time but kept as one at fit time. `overlap` carries the second-largest
    coefficient in KB_MODEL, so that skew shifted every decision.
    """
    return set(_tokenize_si(text))


def _overlap(query_tokens: set[str], context_tokens: set[str]) -> float:
    return (len(query_tokens & context_tokens) / len(query_tokens)) if query_tokens else 0.0


# Logistic-regression knowledge-boundary classifier, fit offline in
# `notebooks/rag/knowledge boundary detection/threshold.ipynb` on 746 labelled
# queries (373 in-syllabus / 373 out-of-syllabus), 5-fold CV + held-out test.
#
# Features combine TWO retrieval modes, not one:
#   - dense_top/dense_avg/overlap: mode="dense" (raw cosine similarity, bounded
#     and interpretable in [0, 1]).
#   - bm25_top/bm25_avg: mode="bm25" (raw BM25 Okapi score, unbounded but
#     StandardScaler-normalized at fit time -- a logistic regression doesn't
#     need a bounded scale the way a fixed threshold rule did).
# A dense-only version of this classifier wrongly rejected 71/373 in-syllabus
# queries (19%) -- almost all proper-noun/date-heavy history questions (kings,
# place names, years) where BM25 keyword matching is strong but a multilingual
# dense embedding doesn't represent the rare Sinhala named entity precisely.
# Adding bm25_top/bm25_avg recovered that signal:
#   dense-only  -> holdout precision=0.879 recall=0.680 f1=0.767 auc=0.883
#   dense+bm25  -> holdout precision=0.881 recall=0.787 f1=0.831 auc=0.900
#
# CV (dense+bm25, mean of 5 folds): precision=0.879 recall=0.845 f1=0.861 auc=0.920
#
# Re-fit by re-running threshold.ipynb end to end and copying the "coef" /
# "intercept" values from `kb_logreg_dense_coefficients.json` into KB_MODEL below.
KB_MODEL = {
    "features": ["dense_top", "dense_avg", "overlap", "bm25_top", "bm25_avg"],
    "coef": [23.64389315115297, -13.660848501739101, 5.933540280649095, 0.16911441161500826, -0.24517224654868158],
    "intercept": -8.97635949584603,
    "probability_threshold": 0.5,
}


def _kb_probability(dense_top: float, dense_avg: float, overlap: float, bm25_top: float, bm25_avg: float) -> float:
    c = KB_MODEL["coef"]
    z = (
        KB_MODEL["intercept"]
        + c[0] * dense_top
        + c[1] * dense_avg
        + c[2] * overlap
        + c[3] * bm25_top
        + c[4] * bm25_avg
    )
    return 1.0 / (1.0 + math.exp(-z))


def knowledge_boundary_detection(query: str, *, top_k: int = 5) -> dict:
    """Decide whether *query* falls within the syllabus boundary.

    Retrieves internally with both mode="dense" and mode="bm25" to match how
    `KB_MODEL` was calibrated. Use `retrieve_context(..., mode="hybrid")`
    separately to build the LLM-facing context -- hybrid retrieval generally
    gives the best recall for answer generation.
    """
    dense_results, dense_context = retrieve_context(query, top_k=top_k, mode="dense")
    bm25_results, _ = retrieve_context(query, top_k=top_k, mode="bm25")

    query_tokens = _boundary_tokens(query)
    context_tokens = _boundary_tokens(dense_context)

    dense_scores = [float(item.get("score", 0.0)) for item in dense_results]
    bm25_scores = [float(item.get("score", 0.0)) for item in bm25_results]

    dense_top = max(dense_scores, default=0.0)
    dense_avg = mean(dense_scores) if dense_scores else 0.0
    overlap = _overlap(query_tokens, context_tokens)
    bm25_top = max(bm25_scores, default=0.0)
    bm25_avg = mean(bm25_scores) if bm25_scores else 0.0

    probability = _kb_probability(dense_top, dense_avg, overlap, bm25_top, bm25_avg)
    within_syllabus = bool(dense_results) and probability >= KB_MODEL["probability_threshold"]

    return {
        "label": "within syllabus" if within_syllabus else "out of syllabus",
        "is_out_of_syllabus": not within_syllabus,
        "confidence": round(probability if within_syllabus else 1.0 - probability, 3),
        "probability_within_syllabus": round(probability, 4),
        "dense_top": round(dense_top, 6),
        "dense_avg": round(dense_avg, 6),
        "overlap": round(overlap, 6),
        "bm25_top": round(bm25_top, 6),
        "bm25_avg": round(bm25_avg, 6),
        "results": dense_results,
        "context_str": dense_context,
        "reason": (
            f"P(within syllabus)={probability:.3f} "
            f"(dense_top={dense_top:.3f}, dense_avg={dense_avg:.3f}, overlap={overlap:.3f}, "
            f"bm25_top={bm25_top:.2f}, bm25_avg={bm25_avg:.2f})"
        ),
    }


def syllabus_boundary_response(query: str, *, top_k: int = 5) -> str:
    """Return a short user-facing syllabus verdict."""
    verdict = knowledge_boundary_detection(query, top_k=top_k)
    if verdict["is_out_of_syllabus"]:
        return f"Out of syllabus: {verdict['reason']}"
    return f"Within syllabus: {verdict['reason']}"


print("knowledge_boundary_detection() ready (logistic regression, dense + BM25 features)")

knowledge_boundary_detection() ready (logistic regression, dense + BM25 features)


<>:7: SyntaxWarning: invalid escape sequence '\w'
<>:7: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_169/975472780.py:7: SyntaxWarning: invalid escape sequence '\w'
  """Tokenize for the `overlap` feature.


In [13]:
QUERY = "පරුමක යනු කවුරුන්ද?"
TOP_K = 5

# Context for answering: hybrid retrieval (best recall for the LLM prompt).
results, context_str = retrieve_context(QUERY, top_k=TOP_K, mode="hybrid")

# Boundary check: logistic-regression classifier on dense-cosine + BM25 features
# (knowledge_boundary_detection retrieves with mode="dense" and mode="bm25" internally).
boundary = knowledge_boundary_detection(QUERY, top_k=TOP_K)

print(boundary["reason"])
print(f"Label : {boundary['label']}  (P={boundary['probability_within_syllabus']:.3f})")
print(f"Query : {QUERY}")
print(f"Chunks (hybrid, used for context): {len(results)}")
print()

for r in results:
    location = " · ".join(filter(None, [
        f"Grade {r['grade']}" if r['grade'] else "",
        r['chapter'], r['section'],
        f"p.{r['page']}" if r['page'] else "",
    ]))
    print(f"  [{r['rank']}] score={r['score']:.5f}  {location}")
    print(f"       {r['text'].replace(chr(10), ' ')}")
    print()

print(context_str)

P(within syllabus)=0.746 (dense_top=0.602, dense_avg=0.573, overlap=0.667, bm25_top=10.59, bm25_avg=8.52)
Label : within syllabus  (P=0.746)
Query : පරුමක යනු කවුරුන්ද?
Chunks (hybrid, used for context): 5

  [1] score=0.03227  Grade 10 · ඓතිහාසික දැනුම හා එහි ප්‍රායෝගික ආදේශනය · කාන්තා නියෝජනය · p.94
       ක්‍රිස්තු පූර්ව දෙවන සියවසේ දී පමණ මෙරට ප්‍රදේශීය පාලනයට සම්බන්ධ ව පරුමක නමින් හැඳින්වූ නායකයෝ පිරිසක් සිටියහ. අනුරාධපුර දිස්ත්‍රික්කයට අයත් බ්‍රාහ්මණයාගම නම් ස්ථානයෙන් සොයා ගෙන තිබෙන එක්තරා සෙල්ලිපියක පරුමකලු සමනා නම් කාන්තාවක් ගැන සඳහන් ය. ඇය පරුමක නදික නමැත්තාගේ බිරිඳ ලෙස එහි සඳහන් ය. මෙහි පරුමකලු යනු පරුමක යන පදයේ ස්ත්‍රීලිං. ස්වරූපය යි. එවක කාන්තාවන් ද ප්‍රාදේශීය පාලන කටයුතුවල නියැලී සිටි බව සනාථ කිරීමට එම සෙල්ලිපිය කදිම සාක්ෂියකි. බෞද්ධ භික්ෂුන් වහන්සේලා උදෙසා ගල් ලෙන් පූජා කිරීමට පෙරමුණ ගත් උපාසිකාවන්ගේ නම් එසේ පූජා කරන ලද ගල්ලෙන්වල කොටා තිබේ. අනුරාධපුර දිස්ත්‍රික්කයට අයත් කොක්ඇබේ නම් ස්ථානයෙන් සොයා ගන්නා ලද ක්‍රිස්තු වර්ෂයෙන් දෙවන සියවසට අයත් පර්වත ලිපියක තලතා ලක්ෂ්මී නම් ව

======================MODULE 1============================

In [14]:
# Installs the packages needed to load and run the model.
%uv pip install -q "transformers>=4.51,<5" accelerate safetensors huggingface_hub hf_transfer

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [15]:
# Configuration — model choice, generation/grounding settings (copied from
# qa-evaluation.ipynb, v6-matched), and the question to answer.
import os
import re
import unicodedata

import torch

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

MODEL_ID = "isji/sinllama-3b-qa-v6-merged"   # or "isji/sinllama-1b-qa-v6-merged"

NO_ANSWER = "මෙම ප්‍රශ්නයට පිළිතුරු දීමට ප්‍රමාණවත් තොරතුරු නොමැත."
MAX_LENGTH = 2048
MAX_NEW_TOKENS = 48
REPETITION_PENALTY = 1.05
GROUNDING_THRESHOLD = 0.50
USE_GROUNDING = True


In [16]:
# Loads the merged model and its tokenizer.
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=model_dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()
model.config.pad_token_id = tokenizer.pad_token_id
print("Model loaded on", model.device)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/19.3M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/860 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.37G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Model loaded on cuda:0


In [17]:
QUERY = "ශ්‍රී ලංකාවේ ප්‍රාග් ඓතිහාසික යුගයේ විසූ මානවයාගේ ජීවන රටාව පිළිබඳව තොරතුරු ලබා ගැනීමට ඉවහල් වන සාධක බොහොමයක් ලැබී ඇති ස්ථානයක් ලෙස සැලකෙනුයේ ?"
TOP_K = 5

In [18]:
# Defines the v6 prompt template and the answer function: build the prompt, generate
# greedily, then apply the grounding gate (refuse if the answer isn't traceable to the
# context) — same logic and thresholds as qa-evaluation.ipynb.
SINHALA_WORD_RE = re.compile(r"[\w඀-෿]+", re.UNICODE)
STOPWORDS = {
    "හා", "සහ", "හෝ", "දී", "ද", "ය", "යි", "වේ", "විය", "වූ", "ලෙස",
    "විසින්", "සඳහා", "සිට", "දක්වා", "එම", "මෙම", "ඒ", "ඔහු", "ඇය",
    "කුමක්ද", "කවුද", "කවදාද", "කෙසේද", "කොපමණද", "මොනවාද",
}

INSTRUCTION = f"""උපදෙස්: පහත සන්දර්භය පමණක් භාවිතා කර ප්‍රශ්නයට පිළිතුරු දෙන්න.
- පිළිතුර සන්දර්භයේ තිබේ නම්, එයින් කෙටිම නිශ්චිත වචන පෙළ පමණක් දෙන්න.
- අමතර පැහැදිලි කිරීම්, පිටත දැනුම හෝ අනුමාන එකතු නොකරන්න.
- සන්දර්භය ප්‍රශ්නයට අදාළ නොවේ නම්, ප්‍රශ්නයට පිළිතුරු දීමට සුදුසු නොවේ නම්, හෝ පිළිතුර සන්දර්භයේ පැහැදිලිව නොමැති නම්, හරියටම මෙය පමණක් දෙන්න: {NO_ANSWER}"""


def clean_text(value):
    text = unicodedata.normalize("NFC", str(value or ""))
    return text.replace("\r\n", "\n").replace("\r", "\n").strip()


def lexical_tokens(value):
    tokens = [t.casefold() for t in SINHALA_WORD_RE.findall(clean_text(value))]
    return [t for t in tokens if len(t) >= 2 and t not in STOPWORDS]


def normalize_answer(value):
    text = clean_text(value).casefold()
    text = re.sub(r"\s+", " ", text)
    return text.strip(" \t\r\n[]{}()<>\"'`.,!?;:।෴")


def is_no_answer(value):
    normalized = normalize_answer(value)
    return normalized == normalize_answer(NO_ANSWER) or "ප්‍රමාණවත් තොරතුරු නොමැත" in normalized


def build_prompt(context, question):
    return (
        f"{INSTRUCTION}\n\n"
        f"සන්දර්භය:\n{clean_text(context)}\n\n"
        f"ප්‍රශ්නය:\n{clean_text(question)}\n\n"
        "පිළිතුර:\n"
    )


def evidence_support(answer, context):
    # Fraction of the answer's content words traceable to the context (a trailing
    # case-ending mismatch of one character is still counted as a match).
    answer_tokens = lexical_tokens(answer)
    if not answer_tokens:
        return 0.0
    normalized_context = " ".join(lexical_tokens(context))

    def supported(token):
        return token in normalized_context or (len(token) >= 4 and token[:-1] in normalized_context)

    return sum(supported(t) for t in answer_tokens) / len(answer_tokens)


def run_qa(context, question):
    prompt = build_prompt(context, question)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=REPETITION_PENALTY,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )

    generated_ids = output_ids[0, inputs["input_ids"].shape[-1]:]
    raw_answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    raw_answer = raw_answer.splitlines()[0].strip(" []{}()<>\"'`") if raw_answer else ""

    if not raw_answer or is_no_answer(raw_answer):
        return {"answer": NO_ANSWER, "raw_answer": raw_answer, "support": 1.0, "gated": False}

    support = evidence_support(raw_answer, context)
    if USE_GROUNDING and support < GROUNDING_THRESHOLD:
        return {"answer": NO_ANSWER, "raw_answer": raw_answer, "support": support, "gated": True}

    return {"answer": raw_answer, "raw_answer": raw_answer, "support": support, "gated": False}


print("run_qa ready.")

run_qa ready.


In [ ]:
# Retrieves context for QUERY (hybrid retrieval) and checks the syllabus-boundary
# classifier, then prints both. retrieve_context() and knowledge_boundary_detection() must
# already be defined/imported (see the notebook intro) before this cell runs.
USE_BOUNDARY_CHECK = True  # False = skip the boundary gate in the next cell, always answer

results, context_str = retrieve_context(QUERY, top_k=TOP_K, mode="hybrid")
boundary = knowledge_boundary_detection(QUERY, top_k=TOP_K)

print(boundary["reason"])
print(f"Label : {boundary['label']}  (P={boundary['probability_within_syllabus']:.3f})")
print(f"Query : {QUERY}")
print(f"Chunks: {len(results)}")
for r in results:
    location = " · ".join(filter(None, [
        f"Grade {r['grade']}" if r.get("grade") else "",
        r.get("chapter"), r.get("section"),
        f"p.{r['page']}" if r.get("page") else "",
    ]))
    print(f"  [{r['rank']}] score={r['score']:.5f}  {location}")
    print(f"       {r['text'].replace(chr(10), ' ')}")

In [ ]:
# Generates and prints the final answer for QUERY — gated by the syllabus-boundary check.
# When USE_BOUNDARY_CHECK is on, run_qa is only called if boundary['label'] is exactly
# "within syllabus"; otherwise the question is treated as out of scope and NO_ANSWER is
# printed directly, without calling run_qa. When the flag is off, the boundary check is
# bypassed and run_qa always runs.
if USE_BOUNDARY_CHECK and boundary["label"] == "out of syllabus":
    answer = NO_ANSWER
    support_note = "n/a (outside syllabus boundary — run_qa not called)"
elif USE_BOUNDARY_CHECK and boundary["label"] == "within syllabus":
    result = run_qa(context_str, QUERY)
    answer = result["answer"]
    support_note = f"{result['support']:.3f}" + ("  (gated to refusal)" if result["gated"] else "")
else:
    result = run_qa(context_str, QUERY)
    answer = result["answer"]
    support_note = f"{result['support']:.3f}" + ("  (gated to refusal)" if result["gated"] else "")

print("Question :", QUERY)
print("Answer   :", answer)
print(f"Support  : {support_note}")